# T3.6 Colab 확인: 평가(`evaluate_model`)의 GPU 결정적 모드 동작과 T4 추론 시간

`src/sr_pipeline/evaluation/evaluator.py`의 `evaluate_model`을 **Colab T4 GPU**(`device="cuda"`)에서 실행해, 로컬 CPU 테스트로는 확인할 수 없는 항목을 봅니다. 로직은 `src/`에 있고 이 노트북은 실행·확인만 합니다.

- 셀 1~2: 저장소(YongMin 브랜치) 가져오기, `lpips` 설치, 최신 코드·GPU 확인
- 셀 3: 점검용 합성 테스트셋 4종과 실행 조건 준비
- 셀 4 **확인 ①** 결정적 알고리즘 강제(`set_model_seed`, 트레이너와 같은 설정) 아래에서 10개 구성(bicubic + 학습하는 9개)을 GPU로 두 번 평가 → 오류 없이 끝나는가, 두 번의 지표가 비트 단위로 같은가, 기록 파일이 생기는가, `psnr_y_pre_rgb`가 B 계열에만 있는가
- 셀 5 **확인 ②** T4 추론 시간: 실제 `infer_benchmark` 조건(320×180 → 1280×720, 배치 1, FP32, 워밍업 10 + 100회)으로 잰 `infer_ms_net`·`infer_ms_total`이 유효한가
- 셀 6 (참고, 통과 기준 아님): 같은 모델의 CPU 평가와 GPU 평가의 지표 차이
- 셀 7 (참고, 선택): Drive의 실제 테스트셋으로 평가 시간·GPU 메모리 (`RUN_REAL_TESTSETS = True`로 바꿨을 때만)
- 셀 8: 요약 (**이 출력을 그대로 복사해 전달**)
- 셀 9 (참고, ①에 오류가 있을 때만): 어떤 연산이 결정적 구현이 없는지 진단

**런타임 유형: T4 GPU.** 셀 1~6, 8은 토큰·Drive가 필요 없습니다(저장소는 Public, 데이터는 합성). 셀 7만 Drive 마운트가 필요합니다. 셀 1~6은 수 분 걸립니다(처음 한 번 AlexNet 가중치를 내려받습니다).

모델은 **학습하지 않은 초기 가중치**입니다(트레이너와 같은 `set_model_seed`·`build_model_and_loss`로 만듦). 이 노트북은 평가 경로가 GPU에서 오류 없이·결정적으로 도는지와 추론 시간만 보며, 지표 값 자체는 의미가 없습니다. 합성 테스트셋과 `pilot_determined` 값은 이 노트북 안에서만 쓰는 **점검용 값**이며 공정 비교 조건이 아닙니다(T4.2에서 확정). `fixed_rules`·`infer_benchmark`는 `configs/fair_comparison.yaml` 값을 그대로 씁니다. `configs/`는 읽기만 합니다.

**다시 재려면** 같은 세션에서 셀 4 → 5 → 8을 실행합니다(셀 4가 이전 결과를 비우고, 각 평가는 자기 출력 폴더를 지운 뒤 씁니다). 평가는 기존 기록을 덮어쓰지 않으므로(`FileExistsError`) 이 정리가 필요합니다.

①②가 모두 OK면 T3.6의 Colab 확인 항목(CUDA 결정적 모드 평가, T4 추론 시간)이 끝납니다. 셀 8 출력을 전달하면 `docs/03-plan.md`에 반영합니다.

In [1]:
import os
REPO_URL = "https://github.com/minyi-k03/VideoProcessing-Project.git"
if os.path.isdir("/content/repo/.git"):
    !git -C /content/repo pull --ff-only          # 이미 받아 둔 경우 최신으로
else:
    !git clone --branch YongMin --depth 1 {REPO_URL} /content/repo
!git -C /content/repo log --oneline -3             # "⚠ 평가 수정(T3.6 리뷰 반영) ..." 이후 커밋이 보여야 함. 안 보이면 로컬에서 git push 먼저
assert os.path.isdir("/content/repo/src/sr_pipeline"), "clone 실패: REPO_URL/푸시 확인"
print("셀 1 OK")

remote: Enumerating objects: 53, done.
remote: Counting objects: 100% (53/53), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 37 (delta 22), reused 36 (delta 21), pack-reused 0 (from 0)
Unpacking objects: 100% (37/37), 27.33 KiB | 1.61 MiB/s, done.
From https://github.com/minyi-k03/VideoProcessing-Project
   cd0c19a..a9455ca  YongMin    -> origin/YongMin
Updating cd0c19a..a9455ca
Fast-forward
 docs/03-plan.md                         |   7 +-
 src/sr_pipeline/data/dataset.py         |  79 ++++++-
 src/sr_pipeline/engine/trainer.py       |   2 +-
 src/sr_pipeline/evaluation/benchmark.py |  82 +++++++-
 src/sr_pipeline/evaluation/evaluator.py | 356 +++++++++++++++++++++++++++++++-
 src/sr_pipeline/evaluation/metrics.py   |  66 +++++-
 tests/test_benchmark.py                 |  65 ++++++
 tests/test_evaluation.py                |  97 ++++++++-
 tests/test_evaluator.py                 | 324 +++++++++++++++++++++++++++++
 tests/test_testset_dataset.py           |  65 ++

In [2]:
import sys, importlib, subprocess, platform, inspect
from importlib.metadata import version as package_version
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lpips"], check=True)   # Colab에 기본 설치돼 있지 않다
sys.path.insert(0, "/content/repo/src")
sys.path_importer_cache.pop("/content/repo/src", None)   # 클론 전에 import를 시도했던 커널의 "경로 없음" 캐시 제거
importlib.invalidate_caches()
# 이전 실행에서 import한 옛 모듈이 커널 메모리에 남아 있으면 git pull 후에도 옛 코드가 실행된다 -> 캐시를 비운다.
for name in [m for m in sys.modules if m == "sr_pipeline" or m.startswith("sr_pipeline.")]:
    del sys.modules[name]
import numpy as np, torch, lpips
from sr_pipeline.evaluation import evaluator
from sr_pipeline.evaluation.benchmark import measure_inference_time
assert (inspect.signature(measure_inference_time).parameters["scale"].default is inspect.Parameter.empty
        and hasattr(evaluator, "_move_from_staging")), \
    "evaluator가 옛 버전입니다: 로컬에서 git push 한 뒤 셀 1을 다시 실행하세요."

assert torch.cuda.is_available(), "GPU 런타임이 아님: 런타임 > 런타임 유형 변경 > T4 GPU"
DEVICE = "cuda"
COMMIT = subprocess.run(["git", "-C", "/content/repo", "rev-parse", "HEAD"],
                        capture_output=True, text=True, check=True).stdout.strip()
print("torch", torch.__version__, "| CUDA", torch.version.cuda, "| cuDNN", torch.backends.cudnn.version(),
      "| lpips", package_version("lpips"), "|", torch.cuda.get_device_name(0))
print("commit", COMMIT)
print("셀 2 OK")

torch 2.14.1+cu130 | CUDA 13.0 | cuDNN 92800 | lpips ? | Tesla T4
commit a9455ca82b9ddb9b01ca8fecdd04b54a5f81e325
셀 2 OK


In [13]:
# 점검용 합성 테스트셋 4종과 실행 조건. pilot_determined 값은 이 노트북에서만 쓰는 점검용 값이다(공정 비교 조건 아님).
# 실제 configs/fair_comparison.yaml을 읽어 pilot_determined만 채운 사본을 /content에 쓰고, resolve_run_config로
# 구성별 실행 조건을 확정한다. fixed_rules(가장자리 4, 데이터셋 4종, LPIPS alex)와 infer_benchmark는 원본 그대로다.
import copy, json, math, shutil, time
from dataclasses import asdict
from pathlib import Path
import yaml
from PIL import Image
from sr_pipeline.common.seed import set_model_seed
from sr_pipeline.config.schema import resolve_run_config
from sr_pipeline.data.prepare import prepare_testsets
from sr_pipeline.engine import trainer
from sr_pipeline.evaluation.benchmark import count_parameters

REPO = Path("/content/repo")
ROOT = Path("/content/t3_6_check")
shutil.rmtree(ROOT, ignore_errors=True)   # 이 노트북의 산출물만 지운다(다시 실행해도 같은 조건에서 시작)

FAIR = yaml.safe_load((REPO / "configs" / "fair_comparison.yaml").read_text(encoding="utf-8"))
RULES = FAIR["fixed_rules"]
DATASETS, SCALE = RULES["datasets"], RULES["scale"]

# HR 영상: 사인파(에지) + 잡음. LPIPS(AlexNet)가 돌 만큼 크게(가장자리를 잘라도 한 변 100화소 이상) 만든다.
HR_SIZES = [(128, 160), (120, 120)]
hr_root = ROOT / "hr_src"
rng = np.random.default_rng(2024)
for name in DATASETS:
    (hr_root / name).mkdir(parents=True)
    for index, (h, w) in enumerate(HR_SIZES):
        yy, xx = np.mgrid[0:h, 0:w]
        base = 127.5 + 60.0 * np.sin(xx / 5.0) + 40.0 * np.cos(yy / 9.0)
        image = np.clip(base[..., None] + rng.normal(0.0, 15.0, (h, w, 3)), 0, 255).astype(np.uint8)
        Image.fromarray(image).save(hr_root / name / f"img{index}.png")
DATA_DIR = ROOT / "data"
prepare_testsets(hr_root, DATA_DIR, SCALE, list(DATASETS), RULES["degradation"], RULES["bicubic_rule"])

CHECK_PILOT = {
    "patch_pool_id": "colab_check_no_pool", "patch_pool_size": 1, "patch_size_hr": 96,
    "patches_per_epoch": 1, "epochs": 1, "batch_size": 1, "learning_rate": 1e-3,
    "optimizer": "adam", "scheduler": "none", "augmentation": "rot90x4_hflip",
    "edge_loss_lambda": 0.1, "edge_top_q": 10,
    # 대표 영상 저장(Before | After | HR, 확대본)도 GPU에서 돌려 보려고 합성 영상 id를 넣는다.
    "representative_images": {name: {"median": "img0", "worst": "img1"} for name in DATASETS},
}
check_fair = copy.deepcopy(FAIR)
check_fair["pilot_determined"] = CHECK_PILOT
CHECK_FAIR_PATH = ROOT / "fair_comparison_check.yaml"
CHECK_FAIR_PATH.write_text(yaml.safe_dump(check_fair, allow_unicode=True), encoding="utf-8")

CONFIGS = ["bicubic", "baseline", "A", "B", "C", "Bp", "AB", "AC", "BC", "ABC"]
RUN_CONFIGS = {cid: resolve_run_config(CHECK_FAIR_PATH, REPO / "configs" / "experiments.yaml", cid, 0) for cid in CONFIGS}
assert [cid for cid, rc in RUN_CONFIGS.items() if not rc.trainable] == ["bicubic"]
METRICS = ("psnr_y", "ssim_y", "psnr_rgb", "edge_psnr", "lpips", "psnr_y_pre_rgb")

def make_model(rc, device=DEVICE):
    # 트레이너(train_run)와 같은 순서: set_model_seed(결정적 알고리즘 강제 포함) -> build_model_and_loss.
    # 학습 없는 구성(bicubic)은 모델이 없다(model=None).
    if not rc.trainable:
        return None
    set_model_seed(rc.seed)
    model, _ = trainer.build_model_and_loss(rc.switches, rc.fair_comparison)
    return model.to(device).eval()

def make_run_meta(rc, model):
    # trainer.train_and_evaluate가 만드는 run_meta와 같은 키. 학습하지 않았으므로 train_minutes=0.
    return {
        "run_id": trainer.build_run_id(COMMIT, rc.config_id, rc.seed), "commit": COMMIT,
        "config_id": rc.config_id, "seed": rc.seed,
        "patch_pool_id": rc.fair_comparison["pilot_determined"]["patch_pool_id"],
        "switches": asdict(rc.switches), "params": 0 if model is None else count_parameters(model),
        "train_minutes": 0.0, "status": "complete",
    }

def evaluate(label, rc, model, data_dir=DATA_DIR, fair=None, device=DEVICE):
    # 실행마다 <ROOT>/<label>/에 summary.csv·per_image.csv·images/를 쓴다. evaluate_model은 기존 기록을 덮어쓰지 않으므로
    # 셀을 다시 실행해도 되게 이 노트북의 점검용 출력 폴더만 먼저 지운다.
    shutil.rmtree(ROOT / label, ignore_errors=True)
    return evaluator.evaluate_model(model, make_run_meta(rc, model), data_dir, ROOT / label,
                                    fair_comparison=fair or rc.fair_comparison, device=device)

results, timings, errors = {}, {}, {}
print("셀 3 OK: 합성 테스트셋", DATASETS, "x", len(HR_SIZES), "장 | 구성", CONFIGS)
print("infer_benchmark:", FAIR["infer_benchmark"])

셀 3 OK: 합성 테스트셋 ['Set5', 'Set14', 'BSD100', 'Urban100'] x 2 장 | 구성 ['bicubic', 'baseline', 'A', 'B', 'C', 'Bp', 'AB', 'AC', 'BC', 'ABC']
infer_benchmark: {'lr_height': 180, 'lr_width': 320, 'sr_height': 720, 'sr_width': 1280, 'batch_size': 1, 'precision': 'fp32', 'warmup_runs': 10, 'repeat_runs': 100}


In [14]:
# 확인 ①: 결정적 알고리즘 강제 아래에서 GPU 평가가 오류 없이 끝나고, 두 번 평가한 지표가 비트 단위로 같은가.
# 트레이너는 학습 시작 때 set_model_seed로 강제를 켜고 같은 프로세스에서 evaluate_model을 부른다. 여기서도 먼저 켠다
# (bicubic은 모델이 없어 make_model이 set_model_seed를 부르지 않으므로).
results.clear(); timings.clear(); errors.clear()   # 다시 실행할 때 이전 결과가 섞이지 않게 비운다
set_model_seed(0)
assert torch.are_deterministic_algorithms_enabled() and os.environ.get("CUBLAS_WORKSPACE_CONFIG")

for cid, rc in RUN_CONFIGS.items():
    try:
        model = make_model(rc)
        s1, p1 = evaluate(f"{cid}_run1", rc, model)
        s2, p2 = evaluate(f"{cid}_run2", rc, model)
    except Exception as error:   # 결정적 구현이 없는 연산이면 RuntimeError가 난다 -> 셀 9로 진단
        errors[cid] = f"{type(error).__name__}: {error}"[:400]
        results[f"① {cid:8s} 강제 모드에서 오류 없이 평가"] = False
        print("FAIL", cid, errors[cid])
        continue
    same = p1 == p2 and all(getattr(a, k) == getattr(b, k) for a, b in zip(s1, s2) for k in METRICS)
    written = all((ROOT / f"{cid}_run1" / n).exists() for n in ("summary.csv", "per_image.csv", "images"))
    pre_rgb_rule = all((s.psnr_y_pre_rgb is not None) == rc.switches.B for s in s1)
    results[f"① {cid:8s} 강제 모드에서 오류 없이 평가"] = True
    results[f"① {cid:8s} 두 번 평가한 지표 비트 단위 동일"] = same
    results[f"① {cid:8s} summary·per_image·대표 영상 기록 생성"] = written
    results[f"① {cid:8s} psnr_y_pre_rgb는 B 계열에만"] = pre_rgb_rule
    timings[cid] = [(s.infer_ms_net, s.infer_ms_total) for s in (s1[0], s2[0])]
    ok = same and written and pre_rgb_rule
    print(f"{'OK  ' if ok else 'FAIL'} {cid:8s} | Set5 PSNR(Y) {s1[0].psnr_y:7.3f} LPIPS {s1[0].lpips:.4f} "
          f"pre_rgb {s1[0].psnr_y_pre_rgb} | 행 {len(s1)}/{len(p1)}")

OK   bicubic  | Set5 PSNR(Y)  29.700 LPIPS 0.6159 pre_rgb None | 행 4/8
OK   baseline | Set5 PSNR(Y)   6.614 LPIPS 1.1158 pre_rgb None | 행 4/8
OK   A        | Set5 PSNR(Y)   6.614 LPIPS 1.1158 pre_rgb None | 행 4/8
OK   B        | Set5 PSNR(Y)   6.627 LPIPS 1.1531 pre_rgb 6.6007877046047625 | 행 4/8
OK   C        | Set5 PSNR(Y)  29.655 LPIPS 0.6061 pre_rgb None | 행 4/8
OK   Bp       | Set5 PSNR(Y)   6.614 LPIPS 1.1158 pre_rgb None | 행 4/8
OK   AB       | Set5 PSNR(Y)   6.627 LPIPS 1.1531 pre_rgb 6.6007877046047625 | 행 4/8
OK   AC       | Set5 PSNR(Y)  29.655 LPIPS 0.6061 pre_rgb None | 행 4/8
OK   BC       | Set5 PSNR(Y)  29.670 LPIPS 0.6059 pre_rgb 29.671063901364015 | 행 4/8
OK   ABC      | Set5 PSNR(Y)  29.670 LPIPS 0.6059 pre_rgb 29.671063901364015 | 행 4/8


In [15]:
# 확인 ②: T4 추론 시간. 확인 ①의 두 번의 평가가 각각 infer_benchmark 조건으로 잰 값을 본다.
# 기준: bicubic은 net == 0, total > 0 / 학습하는 구성은 0 < net, total 유한, net <= total x 1.1(측정 잡음 여유).
# 두 번 잰 값의 차이(%)는 참고용이다(측정 잡음이 얼마나 되는지 가늠).
# 첫 실행에서는 처음 재는 신경망 구성(baseline)의 net 1회차가 GPU가 덜 데워져 높게 나올 수 있다. 그 항목만 FAIL이면
# 같은 세션에서 셀 4 -> 5 -> 8을 다시 실행해 재현되는지 본다.
print(f"{'구성':8s} {'net 1회차':>10s} {'net 2회차':>10s} {'total 1회차':>12s} {'total 2회차':>12s} {'total 차이':>10s}")
for cid, runs in timings.items():
    (n1, t1), (n2, t2) = runs
    if cid == "bicubic":
        ok = n1 == 0.0 and n2 == 0.0 and all(math.isfinite(t) and t > 0 for t in (t1, t2))
    else:
        ok = all(math.isfinite(t) and 0 < n <= t * 1.1 for n, t in runs)
    results[f"② {cid:8s} 추론 시간 유효"] = ok
    spread = abs(t1 - t2) / max(t1, t2) * 100
    print(f"{cid:8s} {n1:10.3f} {n2:10.3f} {t1:12.3f} {t2:12.3f} {spread:9.1f}%  {'OK' if ok else 'FAIL'} (ms)")
results["② 측정 장치가 T4"] = "T4" in torch.cuda.get_device_name(0)
print("측정 조건:", {k: FAIR["infer_benchmark"][k] for k in ("lr_height", "lr_width", "warmup_runs", "repeat_runs")})

구성          net 1회차    net 2회차    total 1회차    total 2회차   total 차이
bicubic       0.000      0.000        0.624        0.608       2.6%  OK (ms)
baseline      1.602      1.532        1.568        1.565       0.2%  OK (ms)
A             1.535      1.547        1.567        1.565       0.1%  OK (ms)
B             1.193      1.187        2.086        2.098       0.6%  OK (ms)
C             1.543      1.554        2.053        2.051       0.1%  OK (ms)
Bp            1.550      1.548        1.571        1.571       0.0%  OK (ms)
AB            1.191      1.193        2.098        2.090       0.3%  OK (ms)
AC            1.549      1.549        2.067        2.074       0.3%  OK (ms)
BC            1.194      1.198        2.269        2.266       0.1%  OK (ms)
ABC           1.193      1.198        2.278        2.271       0.3%  OK (ms)
측정 조건: {'lr_height': 180, 'lr_width': 320, 'warmup_runs': 10, 'repeat_runs': 100}


In [6]:
# 참고(통과 기준 아님): 같은 초기 가중치 모델을 CPU에서 평가해 GPU 지표와 비교한다. 합성곱 구현이 달라 SR이
# 아주 조금 달라지고 8비트 양자화에서 몇 화소가 뒤집힐 수 있으므로, 차이는 작지만 0이 아닐 수 있다.
# CPU에서는 시간 측정이 비교 대상이 아니라서 이 셀의 사본에서만 워밍업·반복 횟수를 줄인다(공정 비교 조건 변경 아님).
import pandas as pd
for cid in ("bicubic", "baseline", "B", "C"):
    rc = RUN_CONFIGS[cid]
    if cid in errors:
        print("건너뜀", cid, "(확인 ① 오류)")
        continue
    cpu_fair = copy.deepcopy(rc.fair_comparison)
    cpu_fair["infer_benchmark"].update(warmup_runs=1, repeat_runs=2)
    _, per_cpu = evaluate(f"{cid}_cpu", rc, make_model(rc, device="cpu"), fair=cpu_fair, device="cpu")
    gpu = pd.read_csv(ROOT / f"{cid}_run1" / "per_image.csv", dtype={"image_id": str})
    cpu = pd.DataFrame([asdict(p) for p in per_cpu])
    assert list(gpu["image_id"]) == list(cpu["image_id"]) and list(gpu["dataset"]) == list(cpu["dataset"])
    diffs = {k: float((gpu[k] - cpu[k]).abs().max()) for k in ("psnr_y", "ssim_y", "psnr_rgb", "edge_psnr", "lpips")}
    print(f"{cid:8s} GPU-CPU 최대 차이 | " + " ".join(f"{k} {v:.2e}" for k, v in diffs.items()))

bicubic  GPU-CPU 최대 차이 | psnr_y 5.97e-07 ssim_y 1.21e-08 psnr_rgb 4.86e-07 edge_psnr 6.95e-07 lpips 1.19e-07
baseline GPU-CPU 최대 차이 | psnr_y 3.03e-07 ssim_y 6.69e-09 psnr_rgb 2.94e-07 edge_psnr 2.44e-07 lpips 3.58e-07
B        GPU-CPU 최대 차이 | psnr_y 3.03e-07 ssim_y 6.73e-09 psnr_rgb 2.94e-07 edge_psnr 2.43e-07 lpips 7.15e-07
C        GPU-CPU 최대 차이 | psnr_y 5.86e-07 ssim_y 1.16e-08 psnr_rgb 4.87e-07 edge_psnr 7.73e-07 lpips 1.19e-07


In [7]:
# 참고(선택, 통과 기준 아님): Drive의 T1.4 압축 파일에서 실제 테스트셋(Set5 5, Set14 14, BSD100 100, Urban100 100장)만
# 풀어, 실제 규모의 평가 시간과 GPU 최대 메모리를 잰다(T4 일정 가늠, 실제 크기에서 LPIPS 동작 확인).
# 실제 영상 id를 모르므로 대표 영상은 null(이미지 저장 생략)로 둔다. 모델은 초기 가중치라 지표 값은 의미 없다.
RUN_REAL_TESTSETS = False   # 실행하려면 True로 바꾼다(Drive 마운트 필요, 수 분 이상 걸림)
ARCHIVE_NAME = "div2k_x4_p96_n10000_s42_e8bc539e.zip"   # T1.4·T1.5에서 만든 파일럿 압축 파일
real_stats = {}
if not RUN_REAL_TESTSETS:
    print("건너뜀: RUN_REAL_TESTSETS = False")
else:
    import glob, zipfile
    from google.colab import drive
    drive.mount("/content/drive")
    matches = glob.glob(f"/content/drive/MyDrive/**/{ARCHIVE_NAME}", recursive=True)
    assert len(matches) == 1, f"MyDrive 아래에서 {ARCHIVE_NAME}을 {len(matches)}개 찾았습니다(1개여야 함)."
    REAL_DIR = Path("/content/t3_6_real")
    shutil.rmtree(REAL_DIR, ignore_errors=True)
    with zipfile.ZipFile(matches[0]) as archive:   # 테스트셋만 푼다(패치 풀은 크고 여기서 필요 없음)
        archive.extractall(REAL_DIR, [m for m in archive.namelist() if "testsets/" in m])
    REAL_DATA = next(p.parent.parent for p in REAL_DIR.rglob("testsets/metadata.json"))
    for cid in ("bicubic", "baseline", "B"):
        rc = RUN_CONFIGS[cid]
        real_fair = copy.deepcopy(rc.fair_comparison)
        real_fair["pilot_determined"]["representative_images"] = {n: {"median": None, "worst": None} for n in DATASETS}
        torch.cuda.reset_peak_memory_stats()
        start = time.perf_counter()
        summaries, per_images = evaluate(f"real_{cid}", rc, make_model(rc), data_dir=REAL_DATA, fair=real_fair)
        torch.cuda.synchronize()
        real_stats[cid] = {"분": (time.perf_counter() - start) / 60, "GPU 최대 메모리 GB": torch.cuda.max_memory_allocated() / 1e9,
                           "영상 수": {s.dataset: sum(p.dataset == s.dataset for p in per_images) for s in summaries}}
        print(cid, real_stats[cid])

건너뜀: RUN_REAL_TESTSETS = False


In [17]:
# 이 출력을 그대로 복사해 전달한다(T3.6 Colab 확인 근거).
from importlib.metadata import version as package_version   # 셀 2를 다시 실행하지 않아도 되게 여기서도 가져온다
for name, ok in results.items():
    print("OK  " if ok else "FAIL", name)
print("전체:", "PASS" if results and all(results.values()) else "FAIL")
print("추론 시간(ms, 1회차 net / total):", {cid: tuple(round(v, 3) for v in runs[0]) for cid, runs in timings.items()})
if errors:
    print("오류:", errors)
if real_stats:
    print("실제 테스트셋(참고):", real_stats)
print("commit  :", COMMIT)
print("device  :", torch.cuda.get_device_name(0))
print("versions: torch", torch.__version__, "| CUDA", torch.version.cuda, "| cuDNN", torch.backends.cudnn.version(),
      "| lpips", package_version("lpips"), "| numpy", np.__version__)
print("platform:", platform.platform())

OK   ① bicubic  강제 모드에서 오류 없이 평가
OK   ① bicubic  두 번 평가한 지표 비트 단위 동일
OK   ① bicubic  summary·per_image·대표 영상 기록 생성
OK   ① bicubic  psnr_y_pre_rgb는 B 계열에만
OK   ① baseline 강제 모드에서 오류 없이 평가
OK   ① baseline 두 번 평가한 지표 비트 단위 동일
OK   ① baseline summary·per_image·대표 영상 기록 생성
OK   ① baseline psnr_y_pre_rgb는 B 계열에만
OK   ① A        강제 모드에서 오류 없이 평가
OK   ① A        두 번 평가한 지표 비트 단위 동일
OK   ① A        summary·per_image·대표 영상 기록 생성
OK   ① A        psnr_y_pre_rgb는 B 계열에만
OK   ① B        강제 모드에서 오류 없이 평가
OK   ① B        두 번 평가한 지표 비트 단위 동일
OK   ① B        summary·per_image·대표 영상 기록 생성
OK   ① B        psnr_y_pre_rgb는 B 계열에만
OK   ① C        강제 모드에서 오류 없이 평가
OK   ① C        두 번 평가한 지표 비트 단위 동일
OK   ① C        summary·per_image·대표 영상 기록 생성
OK   ① C        psnr_y_pre_rgb는 B 계열에만
OK   ① Bp       강제 모드에서 오류 없이 평가
OK   ① Bp       두 번 평가한 지표 비트 단위 동일
OK   ① Bp       summary·per_image·대표 영상 기록 생성
OK   ① Bp       psnr_y_pre_rgb는 B 계열에만
OK   ① AB       강제 모드에서 오류 없이 평가
OK   ① AB       두 번 평가한 지표 비트 단위 동일
OK   ① 

In [9]:
# 참고(①에 오류가 있을 때만): 결정적 알고리즘 강제를 warn_only로 바꿔 다시 평가하고, 결정적 구현이 없는 연산 이름을 모은다.
# 코드를 어떻게 바꿀지는 이 출력을 보고 팀이 정한다. 이 셀은 끝에 강제 모드를 원래대로(warn_only=False) 되돌린다.
import warnings
if not errors:
    print("①에 오류가 없어 진단할 것이 없습니다.")
else:
    torch.use_deterministic_algorithms(True, warn_only=True)
    try:
        for cid in errors:
            rc = RUN_CONFIGS[cid]
            with warnings.catch_warnings(record=True) as caught:
                warnings.simplefilter("always")
                evaluate(f"{cid}_diag", rc, make_model(rc))
            ops = sorted({str(w.message).split(" does not have a deterministic")[0].strip()
                          for w in caught if "determinis" in str(w.message)})
            print(cid, "결정적 구현이 없는 연산:", ops or "(경고 없음: 오류 원인이 결정성이 아님 -> 셀 4의 오류 메시지 확인)")
    finally:
        torch.use_deterministic_algorithms(True)

①에 오류가 없어 진단할 것이 없습니다.
